In [1]:
!rm -rf /content/document_store
!rm -rf /content/.deepeval
!rm -f /content/lightweight_rag_qa_dataset*.csv
!rm -f /content/rag_qa_dataset_with_demo_rag_outputs.csv

In [2]:
!grep -R "__redwood_timeboxed__" /content 2>/dev/null | head

# RAG Pipeline Evaluation Project

## Part 1: Dataset Preparation and Document Store Creation

Steps 1 to 28 prepare the QA dataset, select test questions, create expected answers, and export source documents into the `document_store` folder.

# ETL part

load EnterpriseRAG-Bench from Hugging Face

In [3]:
# Step 1: Install required library
# "datasets" is the Hugging Face library used to download public datasets.
!pip install datasets -q

## 1. Load EnterpriseRAG-Bench Dataset
This section installs required libraries and loads the documents/questions datasets from Hugging Face.

In [4]:
# Step 2: Import required libraries

# load_dataset is used to load datasets from Hugging Face.
from datasets import load_dataset

# pandas is used to convert the dataset into table format.
import pandas as pd


# Step 3: Load EnterpriseRAG-Bench dataset from Hugging Face

# This dataset has two important parts:
# 1. documents  -> company-like documents used as RAG knowledge base
# 2. questions  -> test questions with expected answers and expected document IDs

documents_dataset = load_dataset(
    "onyx-dot-app/EnterpriseRAG-Bench",
    "documents",
    split="test"
)

questions_dataset = load_dataset(
    "onyx-dot-app/EnterpriseRAG-Bench",
    "questions",
    split="test"
)


# Step 4: Convert Hugging Face dataset into pandas DataFrame
# DataFrame means table format, like Excel/CSV.

documents_df = documents_dataset.to_pandas()
questions_df = questions_dataset.to_pandas()


# Step 5: Check how many rows and columns are present

print("Documents dataset shape:", documents_df.shape)
print("Questions dataset shape:", questions_df.shape)


# Step 6: Show column names
# This helps us understand what fields are available.

print("\nDocuments columns:")
print(documents_df.columns.tolist())

print("\nQuestions columns:")
print(questions_df.columns.tolist())


# Step 7: Preview first few rows

print("\nDocuments preview:")
display(documents_df.head())

print("\nQuestions preview:")
display(questions_df.head())

README.md:   0%|          | 0.00/6.93k [00:00<?, ?B/s]

data/documents/test.parquet: reconstructing file:   0%|          |  0.00B / 1.41GB            

data/documents/test.parquet: downloading bytes:           |  0.00B            

Generating test split: 0 examples [00:00, ? examples/s]

data/questions/test.parquet: reconstructing file:   0%|          |  0.00B /  409kB            

...

Generating test split: 0 examples [00:00, ? examples/s]

Documents dataset shape: (511962, 4)
Questions dataset shape: (500, 7)

Documents columns:
['doc_id', 'source_type', 'title', 'content']

Questions columns:
['question_id', 'question_type', 'source_types', 'question', 'expected_doc_ids', 'gold_answer', 'answer_facts']

Documents preview:


,doc_id,source_type,title,content
0,dsid_e54ef48bae78474684a957cf613d47d5,confluence,Runbook: Deploy / Upgrade / Roll Back perf-can...,## Purpose\nThis runbook describes the operati...
1,dsid_229dd48e9b1d466a81ebaffe3ec84469,confluence,Cross-account GPU burst SLO contract and CI li...,Summary\n\nOverview:\nThis document defines th...
2,dsid_aeb0022d62bc43beb6549ba92e5655eb,confluence,Third-Party & Vendor Coordination Playbook for...,Overview\n\nPurpose:\nThis playbook documents ...
3,dsid_926174fc4900408c89c98abde46b7225,confluence,First Production Launch Checklist (Canonical T...,## Purpose\nThis page defines the **canonical ...
4,dsid_d511c6d8daf94f998ce6a3d97462af2d,confluence,Go-Live Runway and Week-3 Stability Playbook,Overview\n\nThis playbook defines the technica...



Questions preview:


,question_id,question_type,source_types,question,expected_doc_ids,gold_answer,answer_facts
0,qst_0001,basic,[github],What are the default size limits for file uplo...,[dsid_ae068ee4aa9640159427cd941bef0238],The default limits are 10 MiB per file (max_fi...,[The default per file upload size limit (max_f...
1,qst_0002,basic,[github],What is the name of the new metric added so SR...,[dsid_9550250a59e74f1bbd5612480b2e7100],The new metric is `stream.timebox_finalized` (...,[The new metric added is named stream.timebox_...
2,qst_0003,basic,[linear],What are the acceptance criteria for the proje...,[dsid_3fd6af404fae48e6b8ea5a57875ef78f],The acceptance criteria are: (1) deliver a sta...,[Deliver a stable design JSON token spec that ...
3,qst_0004,basic,[fireflies],In the meeting about onboarding a SaaS product...,[dsid_6c4c1c875e704f09b4d791d64d7bc7e5],The GCP team said entitlement propagation dela...,[Entitlement propagation delays can occur afte...
4,qst_0005,basic,[gmail],What failover sequence and recovery targets di...,[dsid_8e838ab6a98f4cbcb672d41f210ff89c],MedThink's preferred failover hierarchy is EU ...,[MedThink specified a failover hierarchy of EU...


## 2. Inspect Dataset Structure
This section checks one sample document and one sample question to understand the dataset fields.

In [5]:
# Step 8: Check one full document row
# This helps us understand what one knowledge-base document looks like.

print("One document example:")
display(documents_df.iloc[0])


# Step 9: Check one full question row
# This helps us understand what one test question looks like.

print("One question example:")
display(questions_df.iloc[0])

One document example:


,0
doc_id,dsid_e54ef48bae78474684a957cf613d47d5
source_type,confluence
title,Runbook: Deploy / Upgrade / Roll Back perf-can...
content,## Purpose\nThis runbook describes the operati...


One question example:


,0
question_id,qst_0001
question_type,basic
source_types,[github]
question,What are the default size limits for file uplo...
expected_doc_ids,[dsid_ae068ee4aa9640159427cd941bef0238]
gold_answer,The default limits are 10 MiB per file (max_fi...
answer_facts,[The default per file upload size limit (max_f...


## 3. Select Lightweight QA Test Questions
This section selects 10 basic questions with one clear expected source document.

In [6]:
# Step 10: Select simple/basic questions for our first QA test set

# We are starting with "basic" questions because they are easier to verify.
# Later, we can test complex question types.

basic_questions_df = questions_df[questions_df["question_type"] == "basic"].copy()


# Step 11: Keep only questions that have exactly one expected document
# This makes the first version simple:
# one question -> one correct source document -> one expected answer.

basic_questions_df["expected_doc_count"] = basic_questions_df["expected_doc_ids"].apply(len)

single_doc_questions_df = basic_questions_df[
    basic_questions_df["expected_doc_count"] == 1
].copy()


# Step 12: Select first 10 questions
# We are choosing only 10 because this is the first QA prototype.
# After this works, we can increase the count.

selected_questions_df = single_doc_questions_df.head(10).copy()


# Step 13: Show selected questions

print("Selected questions count:", len(selected_questions_df))

display(
    selected_questions_df[
        [
            "question_id",
            "question_type",
            "question",
            "expected_doc_ids",
            "gold_answer",
            "answer_facts"
        ]
    ]
)

Selected questions count: 10


,question_id,question_type,question,expected_doc_ids,gold_answer,answer_facts
0,qst_0001,basic,What are the default size limits for file uplo...,[dsid_ae068ee4aa9640159427cd941bef0238],The default limits are 10 MiB per file (max_fi...,[The default per file upload size limit (max_f...
1,qst_0002,basic,What is the name of the new metric added so SR...,[dsid_9550250a59e74f1bbd5612480b2e7100],The new metric is `stream.timebox_finalized` (...,[The new metric added is named stream.timebox_...
2,qst_0003,basic,What are the acceptance criteria for the proje...,[dsid_3fd6af404fae48e6b8ea5a57875ef78f],The acceptance criteria are: (1) deliver a sta...,[Deliver a stable design JSON token spec that ...
3,qst_0004,basic,In the meeting about onboarding a SaaS product...,[dsid_6c4c1c875e704f09b4d791d64d7bc7e5],The GCP team said entitlement propagation dela...,[Entitlement propagation delays can occur afte...
4,qst_0005,basic,What failover sequence and recovery targets di...,[dsid_8e838ab6a98f4cbcb672d41f210ff89c],MedThink's preferred failover hierarchy is EU ...,[MedThink specified a failover hierarchy of EU...
5,qst_0006,basic,In the draft spec about extending a routing po...,[dsid_184be937d34a412ab5e61366d54d8ed6],The draft proposes this canonical v1 signal pr...,[The draft spec proposes a canonical v1 priori...
6,qst_0007,basic,In a rolling investigation of a model regressi...,[dsid_72ec4a9962ba43e88acd61abbba1052d],"In the first comparison run, the baseline buil...","[In the first comparison run, the older baseli..."
7,qst_0008,basic,"In the internal shiproom runner notes, what is...",[dsid_5fc2dba9f6ac4af2b49b4f546a4298d0],The rollback plan is considered verified in st...,"[In staging, a verified rollback requires a ti..."
8,qst_0009,basic,"In the EdgePath evaluation email thread, what ...",[dsid_85deb10a652742baaf28af6149600001],Redwood said it wouldn't match the competitor'...,[Redwood did not match the competitors 50 perc...
9,qst_0010,basic,How does the new alerting approach group model...,[dsid_c1a6a71323c04c1ba5445aadea340362],It adds a lightweight token_stage_cohort servi...,[The approach adds a lightweight token_stage_c...


## 4. Create Lightweight QA Dataset
This section maps each selected question to its expected source document and prepares the QA dataset.

In [7]:
# Step 14: Get the matching source document for each selected question

# We create a lookup table using doc_id.
# This helps us quickly find a document by its document ID.

documents_lookup = documents_df.set_index("doc_id")


# Step 15: Create rows for our lightweight QA dataset

lightweight_rows = []

for index, question_row in selected_questions_df.iterrows():

    # Each selected question has exactly one expected document ID.
    expected_doc_id = question_row["expected_doc_ids"][0]

    # Find the matching document from documents_df using that expected_doc_id.
    matching_document = documents_lookup.loc[expected_doc_id]

    # Create one QA test row.
    lightweight_rows.append({
        "test_id": f"RAG_TC_{len(lightweight_rows) + 1:03d}",

        # Unique code created by us for easy tracking.
        "input_code": f"EVL-RAG-{len(lightweight_rows) + 1:03d}",

        # Question details from questions dataset.
        "question_id": question_row["question_id"],
        "question_type": question_row["question_type"],
        "question": question_row["question"],

        # Expected source document details.
        "expected_doc_ids": expected_doc_id,
        "expected_doc_title": matching_document["title"],
        "expected_source_type": matching_document["source_type"],
        "expected_doc_content": str(matching_document["content"]).replace('"__redwood_timeboxed__"', "redwood_timeboxed"),

        # Expected answer details.
        "gold_answer": question_row["gold_answer"],
        "answer_facts": question_row["answer_facts"],

        # These fields will be filled after running the RAG chatbot/model.
        "actual_retrieved_doc_ids": "",
        "actual_output": "",

        # These fields will be filled by our embedding evaluator later.
        "expected_embedding_code": "",
        "actual_embedding_code": "",
        "similarity_score": "",

        # Final QA result fields.
        "faithfulness_check": "Not Checked",
        "hallucination_check": "Not Checked",
        "match_status": "Not Run",
        "remarks": ""
    })


# Step 16: Convert rows into DataFrame

lightweight_rag_df = pd.DataFrame(lightweight_rows)


# Step 17: Preview the lightweight QA dataset

print("Lightweight QA dataset shape:", lightweight_rag_df.shape)

display(lightweight_rag_df.head())

Lightweight QA dataset shape: (10, 20)


,test_id,input_code,question_id,question_type,question,expected_doc_ids,expected_doc_title,expected_source_type,expected_doc_content,gold_answer,answer_facts,actual_retrieved_doc_ids,actual_output,expected_embedding_code,actual_embedding_code,similarity_score,faithfulness_check,hallucination_check,match_status,remarks
0,RAG_TC_001,EVL-RAG-001,qst_0001,basic,What are the default size limits for file uplo...,dsid_ae068ee4aa9640159427cd941bef0238,"add multipart/form-data handling, strict conte...",github,description:\nMotivation: users integrating to...,The default limits are 10 MiB per file (max_fi...,[The default per file upload size limit (max_f...,,,,,,Not Checked,Not Checked,Not Run,
1,RAG_TC_002,EVL-RAG-002,qst_0002,basic,What is the name of the new metric added so SR...,dsid_9550250a59e74f1bbd5612480b2e7100,introduce-server-timebox-and-idempotent-cancel...,github,description:\nContext: production customers ob...,The new metric is `stream.timebox_finalized` (...,[The new metric added is named stream.timebox_...,,,,,,Not Checked,Not Checked,Not Run,
2,RAG_TC_003,EVL-RAG-003,qst_0003,basic,What are the acceptance criteria for the proje...,dsid_3fd6af404fae48e6b8ea5a57875ef78f,Develop interactive tone derivatives and Kappa...,linear,description:\nObjective: Create a deterministi...,The acceptance criteria are: (1) deliver a sta...,[Deliver a stable design JSON token spec that ...,,,,,,Not Checked,Not Checked,Not Run,
3,RAG_TC_004,EVL-RAG-004,qst_0004,basic,In the meeting about onboarding a SaaS product...,dsid_6c4c1c875e704f09b4d791d64d7bc7e5,GCP Marketplace onboarding + billing review (R...,fireflies,summary:\nRedwood and the GCP Marketplace team...,The GCP team said entitlement propagation dela...,[Entitlement propagation delays can occur afte...,,,,,,Not Checked,Not Checked,Not Run,
4,RAG_TC_005,EVL-RAG-005,qst_0005,basic,What failover sequence and recovery targets di...,dsid_8e838ab6a98f4cbcb672d41f210ff89c,Regional fallback priorities & logs — post-cal...,gmail,['From: Rafael Mendes <rafael.mendes@redwoodin...,MedThink's preferred failover hierarchy is EU ...,[MedThink specified a failover hierarchy of EU...,,,,,,Not Checked,Not Checked,Not Run,


In [8]:
# Step 18: Save the lightweight QA dataset as a CSV file

# This CSV is our final ETL output.
# We will use this file for QA testing and embedding comparison later.

lightweight_rag_df.to_csv("lightweight_rag_qa_dataset.csv", index=False)


# Step 19: Confirm the file is saved

print("CSV file created successfully: lightweight_rag_qa_dataset.csv")

CSV file created successfully: lightweight_rag_qa_dataset.csv


In [9]:
# Step 20: Install sentence-transformers
# This library gives us an embedding model.
# Embedding means converting text into numeric meaning code/vector.

!pip install sentence-transformers -q

In [10]:
# Step 21: Load embedding model

# SentenceTransformer is used to convert text into embedding vectors.
from sentence_transformers import SentenceTransformer

# This is a small and commonly used embedding model.
# It converts sentence meaning into a numeric vector.
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


## Demo RAG Pipeline Implementation Plan

This section creates a simple demo RAG pipeline only for testing the QA evaluation workflow.

Implementation flow:

1. Create a `document_store` folder.
2. Export each expected document from the QA dataset into a separate `.txt` file inside `document_store`.
3. Add the document file path back into the dataset as `expected_doc_path`.
4. Demo RAG reads documents from the `document_store` folder.
5. Documents are stored in a LangChain-style vector store.
6. Retrieval returns top-k ranked documents for each question.
7. The top-k retrieved contexts are passed to the local Hugging Face answer-generation model.
8. The generated answer is stored as `actual_output`.
9. DeepEval tracing records the generated answer and retrieved contexts.
10. DeepEval metrics evaluate the RAG flow:
    - Contextual Precision
    - Contextual Recall
    - Answer Relevancy
11. If traced metrics do not expose a score, direct `metric.measure()` is used only as separate debug evidence.

Note:

This is not a production RAG chatbot. It is a QA test setup created to generate actual outputs and validate the RAG evaluation workflow end-to-end.


In [11]:
# Prepare QA dataframe for demo RAG pipeline

# Use the lightweight QA dataframe created in the ETL section.
# This avoids reloading the old custom embedding CSV.

qa_df = lightweight_rag_df.copy()

# actual_output will be filled by the demo RAG pipeline.
qa_df["actual_output"] = qa_df["actual_output"].astype("object")

print("qa_df prepared:", qa_df.shape)


qa_df prepared: (10, 20)


In [12]:
# Step 27: Create document_store folder and export expected documents as .txt files

# This step separates document content from the CSV.
# CSV will keep QA test case details.
# document_store folder will keep source documents like a real RAG knowledge base.

import os
import re

# Create document_store folder if it does not already exist
document_store_path = "document_store"
os.makedirs(document_store_path, exist_ok=True)


def clean_filename(text):
    """
    Create a safe file name from document id/title.
    This removes characters that are not safe for file names.
    """
    text = str(text)
    text = re.sub(r"[^a-zA-Z0-9_-]", "_", text)
    return text[:120]


# Create a new column to store each exported document file path
qa_df["expected_doc_path"] = ""

# Export each expected document content into a separate text file
for index, row in qa_df.iterrows():
    test_id = row["test_id"]
    doc_id = row["expected_doc_ids"]
    doc_title = row["expected_doc_title"]
    doc_content = row["expected_doc_content"]

    # Create readable and unique file name
    file_name = f"{test_id}_{clean_filename(doc_id)}.txt"
    file_path = os.path.join(document_store_path, file_name)

    # Write document content into .txt file
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(f"doc_id: {doc_id}\n")
        file.write(f"title: {doc_title}\n\n")
        file.write(str(doc_content))

    # Store file path back into dataframe
    qa_df.loc[index, "expected_doc_path"] = file_path


print("Document store created successfully.")
print("Total documents exported:", len(qa_df))
print("Folder name:", document_store_path)

display(
    qa_df[
        [
            "test_id",
            "expected_doc_ids",
            "expected_doc_title",
            "expected_doc_path"
        ]
    ]
)

Document store created successfully.
Total documents exported: 10
Folder name: document_store


,test_id,expected_doc_ids,expected_doc_title,expected_doc_path
0,RAG_TC_001,dsid_ae068ee4aa9640159427cd941bef0238,"add multipart/form-data handling, strict conte...",document_store/RAG_TC_001_dsid_ae068ee4aa96401...
1,RAG_TC_002,dsid_9550250a59e74f1bbd5612480b2e7100,introduce-server-timebox-and-idempotent-cancel...,document_store/RAG_TC_002_dsid_9550250a59e74f1...
2,RAG_TC_003,dsid_3fd6af404fae48e6b8ea5a57875ef78f,Develop interactive tone derivatives and Kappa...,document_store/RAG_TC_003_dsid_3fd6af404fae48e...
3,RAG_TC_004,dsid_6c4c1c875e704f09b4d791d64d7bc7e5,GCP Marketplace onboarding + billing review (R...,document_store/RAG_TC_004_dsid_6c4c1c875e704f0...
4,RAG_TC_005,dsid_8e838ab6a98f4cbcb672d41f210ff89c,Regional fallback priorities & logs — post-cal...,document_store/RAG_TC_005_dsid_8e838ab6a98f4cb...
5,RAG_TC_006,dsid_184be937d34a412ab5e61366d54d8ed6,Draft Spec: Policy Engine Extensions for Regio...,document_store/RAG_TC_006_dsid_184be937d34a412...
6,RAG_TC_007,dsid_72ec4a9962ba43e88acd61abbba1052d,rolling-bias-bisection-log-jared,document_store/RAG_TC_007_dsid_72ec4a9962ba43e...
7,RAG_TC_008,dsid_5fc2dba9f6ac4af2b49b4f546a4298d0,Shiproom runner: personal prep log and sticky ...,document_store/RAG_TC_008_dsid_5fc2dba9f6ac4af...
8,RAG_TC_009,dsid_85deb10a652742baaf28af6149600001,Licensing offsets & packaging for potential mi...,document_store/RAG_TC_009_dsid_85deb10a652742b...
9,RAG_TC_010,dsid_c1a6a71323c04c1ba5445aadea340362,introduce-token-stage-cohorting-and-route-matr...,document_store/RAG_TC_010_dsid_c1a6a71323c04c1...


In [13]:
# Step 28: Check exported documents inside document_store folder

# This confirms that our document files are created properly.
# The demo RAG system will read documents from this folder.

exported_files = os.listdir(document_store_path)

print("Total files in document_store:", len(exported_files))
print("First few files:")

for file_name in exported_files[:5]:
    print(file_name)

Total files in document_store: 10
First few files:
RAG_TC_008_dsid_5fc2dba9f6ac4af2b49b4f546a4298d0.txt
RAG_TC_004_dsid_6c4c1c875e704f09b4d791d64d7bc7e5.txt
RAG_TC_009_dsid_85deb10a652742baaf28af6149600001.txt
RAG_TC_006_dsid_184be937d34a412ab5e61366d54d8ed6.txt
RAG_TC_002_dsid_9550250a59e74f1bbd5612480b2e7100.txt


## Part 2: Demo RAG Runtime Pipeline

From Step 29 onward, the notebook reads documents from `document_store`, stores documents in a vector store, retrieves top-k documents, generates answers, and runs evaluation checks.

In [14]:
# Step 29: Load documents from document_store folder

# This step reads the exported .txt files.
# Now the demo RAG system will use file-based documents instead of reading document content directly from CSV.

document_store = []

for file_name in os.listdir(document_store_path):
    if file_name.endswith(".txt"):
        file_path = os.path.join(document_store_path, file_name)

        with open(file_path, "r", encoding="utf-8") as file:
            content = file.read()

        document_store.append(
            {
                "file_name": file_name,
                "file_path": file_path,
                "content": content
            }
        )

print("Documents loaded from document_store:", len(document_store))

# Show one loaded document sample
print("Sample file:", document_store[0]["file_name"])
print("Sample content preview:")
print(document_store[0]["content"][:500])

Documents loaded from document_store: 10
Sample file: RAG_TC_008_dsid_5fc2dba9f6ac4af2b49b4f546a4298d0.txt
Sample content preview:
doc_id: dsid_5fc2dba9f6ac4af2b49b4f546a4298d0
title: Shiproom runner: personal prep log and sticky notes

Purpose: quick, scannable personal notes for running the shiproom and weekly readiness sync. Organized as prep items, live-run cues, and follow-up actions. Not a polished playbook — my sticky running list + prompts to escalate.

DAY-BEFORE / 24-48H CHECKS (quick hits)
- Validate Dedicated pool health dashboard (console > Dedicated > Pool-5). Look for 90th pct latency spike > 20% vs baseline.


In [15]:
# Step 30A: Create LangChain vector store retrieval logic

# This step replaces the old manual retrieval logic with professional vector-store retrieval.
#
# Flow:
# 1. Convert loaded documents into LangChain Document objects.
# 2. Store those documents inside an in-memory vector store.
# 3. Use similarity_search_with_score() to retrieve top-k relevant documents.
#
# This follows the same RAG retrieval idea:
# question -> vector store -> top-k relevant documents


!pip install langchain-core -q

from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.embeddings import Embeddings


class SentenceTransformerEmbeddings(Embeddings):
    """
    This wrapper lets LangChain use our existing SentenceTransformer embedding model.
    """

    def __init__(self, embedding_model):
        self.embedding_model = embedding_model

    def embed_documents(self, texts):
        """
        Converts multiple document texts into embeddings.
        """
        embeddings = self.embedding_model.encode(list(texts))
        return embeddings.tolist()

    def embed_query(self, text):
        """
        Converts one user question into an embedding.
        """
        embedding = self.embedding_model.encode(str(text))
        return embedding.tolist()


def extract_doc_id_from_file_content(content):
    """
    Extracts doc_id from the first line of exported document file.
    """
    first_line = str(content).split("\n")[0]

    if first_line.startswith("doc_id:"):
        return first_line.replace("doc_id:", "").strip()

    return ""


# Convert loaded documents into LangChain Document objects.
langchain_documents = []

for document in document_store:
    doc_id = extract_doc_id_from_file_content(document["content"])

    langchain_documents.append(
        Document(
            page_content=document["content"],
            metadata={
                "doc_id": doc_id,
                "file_name": document["file_name"],
                "file_path": document["file_path"]
            }
        )
    )


# Create LangChain embedding wrapper using existing embedding_model.
langchain_embedding_model = SentenceTransformerEmbeddings(embedding_model)


# Create in-memory vector store.
vector_store = InMemoryVectorStore(
    embedding=langchain_embedding_model
)


# Add documents into vector store.
vector_store.add_documents(langchain_documents)


def vector_retrieve(question, document_store=None, document_embeddings=None, top_k=2):
    """
    Retrieves top-k relevant documents using LangChain vector store.

    Input:
    - question: user question
    - top_k: number of documents to retrieve

    Output:
    - best_document: rank 1 document
    - top_k_results: top-k documents in ranked order
    """

    search_results = vector_store.similarity_search_with_score(
        query=question,
        k=top_k
    )

    retrieval_results = []

    for rank_index, result in enumerate(search_results):
        retrieved_document = result[0]
        similarity_score = result[1]

        retrieval_results.append(
            {
                "rank": rank_index + 1,
                "file_name": retrieved_document.metadata["file_name"],
                "file_path": retrieved_document.metadata["file_path"],
                "content": retrieved_document.page_content,
                "similarity_score": round(float(similarity_score), 4)
            }
        )

    best_document = retrieval_results[0]

    return best_document, retrieval_results


# Test LangChain vector store retrieval for first question.
sample_question = qa_df.loc[0, "question"]

best_document, top_k_results = vector_retrieve(
    question=sample_question,
    top_k=2
)

print("Question:", sample_question)
print("Best retrieved document:", best_document["file_name"])
print("Top-k retrieved documents:")

for result in top_k_results:
    print(
        "Rank:", result["rank"],
        "| File:", result["file_name"],
        "| Score:", result["similarity_score"]
    )

Question: What are the default size limits for file uploads and total request size for the new multipart upload support on the OpenAI-compatible API endpoints?
Best retrieved document: RAG_TC_001_dsid_ae068ee4aa9640159427cd941bef0238.txt
Top-k retrieved documents:
Rank: 1 | File: RAG_TC_001_dsid_ae068ee4aa9640159427cd941bef0238.txt | Score: 0.7318
Rank: 2 | File: RAG_TC_002_dsid_9550250a59e74f1bbd5612480b2e7100.txt | Score: 0.3154


### Model Usage in This RAG QA Evaluation Notebook

This notebook uses two different models for two different purposes.

1. **Local Hugging Face model for answer generation**

   The local Hugging Face model is used to generate the demo RAG `actual_output`.

   This is done in:

   ```text
   Step 31: Generate clean demo RAG response
   Method: extract_relevant_answer()
   ```

   Flow:

   ```text
   Question + Top-k retrieved documents
   → Local Hugging Face model
   → Generated answer / actual_output
   ```

   In this notebook, “local model” means the model is downloaded from Hugging Face into the Colab runtime and runs inside Colab.

2. **Groq model with DeepEval for evaluation**

   Groq is used as the evaluation judge through DeepEval.

   This is done in:

   ```text
   DeepEval Contextual Precision section
   Method: ContextualPrecisionMetric()
   ```

   Flow:

   ```text
   Question + Gold answer + Top-k retrieved documents
   → DeepEval Contextual Precision metric using Groq
   → Evaluation score and reason
   ```

Simple meaning:

```text
Hugging Face model = generates the demo RAG answer
Groq + DeepEval = evaluates the RAG retrieval/context quality
```

In [16]:
# Step 31: Generate clean demo RAG response from retrieved document

# This step creates a clean actual_output from the retrieved document.
#
# Existing flow is kept:
# question -> retrieve best document -> generate clean response from that document
#
# Output format:
# Answer: <generated answer text>
#
# Example:
# Answer: The default file upload limit is 10 MiB per file, and the total request size limit is 50 MiB.

!pip install transformers torch -q

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch


def extract_doc_id_from_file_content(content):
    """
    Extracts doc_id from the retrieved document content.
    """
    first_line = str(content).split("\n")[0]

    if first_line.startswith("doc_id:"):
        return first_line.replace("doc_id:", "").strip()

    return ""


def clean_document_text_for_answer_generation(text):
    """
    Creates a cleaned temporary copy of retrieved document text.
    Original document content is not changed.
    """
    lines = str(text).splitlines()

    cleaned_lines = []

    for line in lines:
        line = line.strip()

        if line.startswith("doc_id:"):
            continue

        if line.startswith("title:"):
            continue

        if line == "":
            continue

        cleaned_lines.append(line)

    return " ".join(cleaned_lines)


# Load local instruction/text generation model.
# This model generates a clean answer from question + retrieved document.
#rag_response_model_name = "google/flan-t5-base"
rag_response_model_name = "google/flan-t5-large"

rag_tokenizer = AutoTokenizer.from_pretrained(rag_response_model_name)
rag_response_model = AutoModelForSeq2SeqLM.from_pretrained(rag_response_model_name)



def extract_relevant_answer(question, retrieved_contexts, top_n=3):
    """
    Generates a clean demo RAG response from the retrieved document.

    Input:
    - question: user question
    - document_text: retrieved document content
    - top_n: kept only for Step 32 compatibility

    Output:
    - clean answer with source document id
    """

    cleaned_contexts = [
        clean_document_text_for_answer_generation(context)
        for context in retrieved_contexts
    ]

    context = "\n\n".join(cleaned_contexts)[:2500]

    prompt = f"""
    You are a RAG assistant.

    Use the context to answer the question.
    Write the answer in 1 to 3 complete sentences.
    Include all important numbers, names, steps, or conditions needed to answer the question.
    Do not answer with only one word or a short phrase.
    Do not include unrelated context.

    Question:
    {question}

    Context:
    {context}

    Final Answer:
    """

    inputs = rag_tokenizer(
        prompt,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    )

    with torch.no_grad():
        outputs = rag_response_model.generate(
            **inputs,
            max_new_tokens=120,
            num_beams=4,
            early_stopping=True
        )

    answer = rag_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

    if answer == "":
        answer = "Answer not found in retrieved document."

    return f"Answer: {answer}"


best_document, top_k_results = vector_retrieve(
    question=sample_question,
    top_k=2
)

sample_top_k_contexts = [
    document["content"]
    for document in top_k_results
]

demo_answer = extract_relevant_answer(
    sample_question,
    sample_top_k_contexts,
    top_n=3
)

print("Generated demo answer:")
print(demo_answer)

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generated demo answer:
Answer: 10MiB, 50MiB, 20.


In [17]:
# Step 32: Generate actual output using traced RAG pipeline

# This step runs the demo RAG pipeline for every question.
#
# Final flow:
# 1. Loop through every question in qa_df.
# 2. Use vector_retrieve() to retrieve top-k relevant documents.
# 3. Send original question + top-k contents to extract_relevant_answer().
# 4. Store actual_output in CSV for final report.
# 5. Store retrieved contexts inside DeepEval trace for professional evaluation style.

!pip install deepeval -q

from deepeval.tracing import observe, update_current_trace


@observe(name="rag_pipeline_run")
def run_traced_rag_pipeline(question, expected_output=None, top_k=2):
    """
    Runs one complete RAG flow and records important data into DeepEval trace.

    This function does:
    - retrieves top-k documents
    - generates answer using Step 31 function
    - stores actual output and retrieved contexts into DeepEval trace

    Input:
    - question: user question
    - expected_output: gold_answer used by DeepEval metrics
    - top_k: number of documents to retrieve

    Output:
    - generated_answer: answer generated by demo RAG model
    """

    # Step 1: Retrieve top-k relevant documents using LangChain vector store.
    best_document, top_k_results = vector_retrieve(
        question=question,
        top_k=top_k
    )

    # Step 2: Extract retrieved document contents in ranked order.
    top_k_contexts = [
        document["content"]
        for document in top_k_results
    ]

    # Step 3: Generate answer using Step 31 answer-generation function.
    generated_answer = extract_relevant_answer(
        question,
        top_k_contexts,
        top_n=top_k
    )

    # Step 4: Store RAG execution details inside DeepEval trace.
    # Full retrieved contexts are kept in trace, not in CSV.
    update_current_trace(
        output=generated_answer,
        expected_output=expected_output,
        retrieval_context=top_k_contexts
    )

    return generated_answer


# Create only final answer column.
qa_df["actual_output"] = ""


# Run traced RAG pipeline for every test case.
for index, row in qa_df.iterrows():

    generated_answer = run_traced_rag_pipeline(
        question=row["question"],
        expected_output=row["gold_answer"],
        top_k=2
    )

    # Store only final answer in CSV.
    # Retrieved contexts are stored in DeepEval trace.
    qa_df.loc[index, "actual_output"] = generated_answer


# Display updated output.
display(
    qa_df[
        [
            "test_id",
            "question",
            "expected_doc_ids",
            "actual_output"
        ]
    ]
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.7/567.7 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 1.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.28.0 requires click<9.0.0,>=8.4.2, but you have click 8.3.3 which is incompatible.


[Confident AI Trace Log]  No Confident AI API key found. Skipping trace posting.

[Confident AI Trace Log]  No Confident AI API key found. Skipping trace posting.

[Confident AI Trace Log]  No Confident AI API key found. Skipping trace posting.

[Confident AI Trace Log]  No Confident AI API key found. Skipping trace posting.

...

[Confident AI Trace Log]  No Confident AI API key found. Skipping trace posting.

[Confident AI Trace Log]  No Confident AI API key found. Skipping trace posting.

[Confident AI Trace Log]  No Confident AI API key found. Skipping trace posting.

[Confident AI Trace Log]  No Confident AI API key found. Skipping trace posting.

,test_id,question,expected_doc_ids,actual_output
0,RAG_TC_001,What are the default size limits for file uplo...,dsid_ae068ee4aa9640159427cd941bef0238,"Answer: 10MiB, 50MiB, 20."
1,RAG_TC_002,What is the name of the new metric added so SR...,dsid_9550250a59e74f1bbd5612480b2e7100,Answer: timeboxed
2,RAG_TC_003,What are the acceptance criteria for the proje...,dsid_3fd6af404fae48e6b8ea5a57875ef78f,Answer: Implementation should use runtime CSS ...
3,RAG_TC_004,In the meeting about onboarding a SaaS product...,dsid_6c4c1c875e704f09b4d791d64d7bc7e5,Answer: GCP emphasized keeping dimension names...
4,RAG_TC_005,What failover sequence and recovery targets di...,dsid_8e838ab6a98f4cbcb672d41f210ff89c,"Answer: No-op, failover, partial shift, rollba..."
5,RAG_TC_006,In the draft spec about extending a routing po...,dsid_184be937d34a412ab5e61366d54d8ed6,Answer: Reachability: ok|degraded|down + confi...
6,RAG_TC_007,In a rolling investigation of a model regressi...,dsid_72ec4a9962ba43e88acd61abbba1052d,Answer: The average triage rubric score change...
7,RAG_TC_008,"In the internal shiproom runner notes, what is...",dsid_5fc2dba9f6ac4af2b49b4f546a4298d0,Answer: 10m.
8,RAG_TC_009,"In the EdgePath evaluation email thread, what ...",dsid_85deb10a652742baaf28af6149600001,Answer: Prepay bucket + seat licenses with an ...
9,RAG_TC_010,How does the new alerting approach group model...,dsid_c1a6a71323c04c1ba5445aadea340362,Answer: Introduces an alert route-matrix gener...


"""
## 8. Retrieval Document Check
This section checks whether the RAG system retrieved the expected source document.
"""


In [18]:
# Groq model setup for DeepEval using custom wrapper
# This lets DeepEval use Groq instead of OpenAI.
# This also avoids Groq tool-call wrapper format issues.

!pip install deepeval groq -q

import os
import re
from google.colab import userdata
from groq import Groq
from deepeval.models import DeepEvalBaseLLM


os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")


class GroqDeepEvalModel(DeepEvalBaseLLM):

    #def __init__(self, model_name="llama-3.3-70b-versatile"):
    #def __init__(self, model_name="llama-3.1-8b-instant"):
    def __init__(self, model_name="openai/gpt-oss-20b"):
        self.model_name = model_name
        self.client = Groq(api_key=os.environ["GROQ_API_KEY"])

    def load_model(self):
        return self.client

    def clean_response(self, text):
        text = str(text).strip()

        # Remove Groq tool-call wrapper if Groq returns it.
        text = text.replace("<function=json_tool_call>", "")
        text = text.replace("</function>", "")

        # Keep only JSON content if extra text is present.
        json_match = re.search(r"(\{.*\}|\[.*\])", text, re.DOTALL)

        if json_match:
            return json_match.group(1).strip()

        return text

    def generate(self, prompt: str, schema=None, **kwargs):
        try:
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=[
                    {
                        "role": "system",
                        "content": "Return only valid JSON. Do not use tool calls. Do not wrap JSON in function tags. Escape all double quotes inside string values using backslash. Do not copy raw quoted text into JSON strings."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0,
                response_format={"type": "json_object"},
                max_completion_tokens=2048
            )

        except Exception as error:
            raise Exception(f"Groq evaluator error: {str(error)}")


        cleaned_text = self.clean_response(response.choices[0].message.content)

        if schema is not None:
            return schema.model_validate_json(cleaned_text)

        return cleaned_text

    async def a_generate(self, prompt: str, **kwargs) -> str:
        return self.generate(prompt, **kwargs)

    def get_model_name(self):
        return self.model_name


groq_model = GroqDeepEvalModel()

print("Groq custom model connected for DeepEval.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.8 MB/s eta 0:00:00
Groq custom model connected for DeepEval.


Component-Level Evaluation for RAG Functions

In [19]:
## Component-Level Evaluation for RAG Functions

# This section tests internal RAG components separately.
#
# Component 1:
# vector_retrieve() -> checks retriever quality
#
# Component 2:
# extract_relevant_answer() -> checks generator answer quality
#
# Existing RAG functions are not changed.
# We create wrapper functions only for component-level evaluation.

!pip install deepeval -q

from deepeval.tracing import observe, update_current_span, update_current_trace
from deepeval.test_case import LLMTestCase
from deepeval.dataset import Golden, EvaluationDataset
from deepeval.evaluate import ErrorConfig, AsyncConfig
from deepeval.metrics import (
    ContextualRelevancyMetric,
    AnswerRelevancyMetric,
    FaithfulnessMetric
)

import time


# Metric for retriever component.
component_contextual_relevancy_metric = ContextualRelevancyMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True,
    verbose_mode=True
)


# Metrics for generator component.
component_answer_relevancy_metric = AnswerRelevancyMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True,
    verbose_mode=True
)

component_faithfulness_metric = FaithfulnessMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True,
    verbose_mode=True
)


@observe(name="retriever_component", metrics=[component_contextual_relevancy_metric])
def component_vector_retrieve(question, top_k=2):
    best_document, top_k_results = vector_retrieve(
        question=question,
        top_k=top_k
    )

    retrieval_context = [
        document["content"][:2500]
        for document in top_k_results
    ]


    update_current_span(
        test_case=LLMTestCase(
            input=question,
            actual_output="Retrieved top-k documents for the question.",
            retrieval_context=retrieval_context
        )
    )

    return best_document, top_k_results, retrieval_context


@observe(
    name="generator_component",
    metrics=[
        component_answer_relevancy_metric,
        component_faithfulness_metric
    ]
)
def component_extract_relevant_answer(question, retrieval_context):
    generated_answer = extract_relevant_answer(
        question=question,
        retrieved_contexts=retrieval_context,
        top_n=len(retrieval_context)
    )

    update_current_span(
        test_case=LLMTestCase(
            input=question,
            actual_output=generated_answer,
            retrieval_context=retrieval_context
        )
    )

    return generated_answer


@observe(name="component_level_rag_pipeline")
def run_component_level_rag_pipeline(question, expected_output=None, top_k=2):
    best_document, top_k_results, retrieval_context = component_vector_retrieve(
        question=question,
        top_k=top_k
    )

    generated_answer = component_extract_relevant_answer(
        question=question,
        retrieval_context=retrieval_context
    )

    update_current_trace(
        input=question,
        output=generated_answer,
        expected_output=expected_output,
        retrieval_context=retrieval_context
    )

    return generated_answer


component_goldens = []

for row_index, row in qa_df.iterrows():
    component_goldens.append(
        Golden(
            input=row["question"],
            expected_output=row["gold_answer"]
        )
    )


component_evaluation_dataset = EvaluationDataset(
    goldens=component_goldens
)


for row_index, golden in enumerate(
    component_evaluation_dataset.evals_iterator(
        metrics=[],
        error_config=ErrorConfig(ignore_errors=True),
        async_config=AsyncConfig(run_async=False)
    )
):

    run_component_level_rag_pipeline(
        question=golden.input,
        expected_output=golden.expected_output,
        top_k=2
    )

    print("Component-level evaluation completed:", qa_df.loc[row_index, "test_id"])

    time.sleep(60)

Output()

Component-level evaluation completed: RAG_TC_001

**************************************************

Answer Relevancy Verbose Logs

**************************************************

...

Component-level evaluation completed: RAG_TC_009

Component-level evaluation completed: RAG_TC_010

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:              What are the default size limits for file uploads and total request size for the     │
│  │                         new multipart upload support on the OpenAI-compatible API endpoints?                 │
│  │     Actual Output:      Answer: 10MiB, max_total_request_size (default 50MiB), max_parts_count (default      │
│  │                         20).                                                                                 │
│  │     Expected Output:    The default limits are 10 MiB per file (max_file_size) and 50 MiB total per          │
│  │                         request (max_total_request_size) for multipart uploads on the OpenAI-compatible      │
│  │                         endpoints.                                                                           │
│  └── Metrics                                                                                                    │
│             Status       ┃ Metric             ┃ Score           ┃ Threshold                ┃ Reason             │
│      ━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_2                                                                                                 │
│  ├──   Input:              What is the name of the new metric added so SRE can track when server-side           │
│  │                         streaming sessions get finalized due to hitting the time limit?                      │
│  │     Actual Output:      Answer: token cost                                                                   │
│  │     Expected Output:    The new metric is `stream.timebox_finalized` (with labels for route and model).      │
│  └── Metrics                                                                                                    │
│             Status       ┃ Metric             ┃ Score           ┃ Threshold                ┃ Reason             │
│      ━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_3                                                                                                 │
│  ├──   Input:              What are the acceptance criteria for the project introducing an algorithm to         │
│  │                         generate interactive UI color s

⚠ WARNING: No hyperparameters logged.
» ]8;id=26507;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 1228.22s | token cost: None)
» Test Results (10 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 10

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

### How Tracing Supports DeepEval RAG Metrics

```text
Retriever finds top-k similar documents.
Tracing records those retrieved documents and generated answer.
DeepEval metrics use the traced data for evaluation.
```


In [ ]:
## 9. Contextual Precision and Answer Relevancy Check using DeepEval Trace

# This section uses DeepEval's professional RAG metric with tracing.
#
# Contextual Precision checks whether useful retrieved contexts
# are ranked higher than less useful/noisy contexts.
#
# In this traced version:
# - Step 32 traced function stores retrieval_context inside DeepEval trace.
# - This cell provides input and expected_output using Golden.
# - DeepEval reads retrieval_context from the trace during evaluation.

!pip install deepeval -q

from deepeval.evaluate import ErrorConfig, AsyncConfig
import json
import os
from deepeval.dataset import Golden, EvaluationDataset
from deepeval.metrics import ContextualPrecisionMetric, AnswerRelevancyMetric


# Create DeepEval Contextual Precision metric.
# Groq is used as the evaluation judge through our custom wrapper.
contextual_precision_metric = ContextualPrecisionMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True,
    verbose_mode=True
)


answer_relevancy_metric = AnswerRelevancyMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True,
    verbose_mode=True
)

# Create Golden test cases from qa_df.
# Each Golden contains:
# - input: original question
# - expected_output: gold_answer / expected answer
goldens = []

for row_index, row in qa_df.iterrows():
    goldens.append(
        Golden(
            input=row["question"],
            expected_output=row["gold_answer"]
        )
    )


# Create DeepEval evaluation dataset.
# This lets DeepEval run each Golden test case one by one.
evaluation_dataset = EvaluationDataset(
    goldens=goldens
)


# Run Contextual Precision using traced RAG function.
# During each run:
# - evals_iterator provides the current Golden
# - run_traced_rag_pipeline() retrieves top-k docs and stores them in trace
# - ContextualPrecisionMetric reads retrieval_context from the trace
import time

#for row_index, golden in enumerate(
#    evaluation_dataset.evals_iterator(metrics=[contextual_precision_metric])
#):

# Store only error reasons when errors happen.
#error_reason_file = "rag_metric_error_reasons.json"
error_reason_file = "/content/rag_metric_error_reasons.json"
error_reasons = {}

for row_index, golden in enumerate(
    evaluation_dataset.evals_iterator(
        metrics=[
            contextual_precision_metric,
            answer_relevancy_metric
        ],
        error_config=ErrorConfig(ignore_errors=True),
        async_config=AsyncConfig(run_async=False)
    )
):



    try:
        # Run traced RAG pipeline for this test case.
        # This call creates actual_output and retrieval_context inside DeepEval trace.
        generated_answer = run_traced_rag_pipeline(
            question=golden.input,
            expected_output=golden.expected_output,
            top_k=2
        )

        # Store actual output in CSV/report.
        qa_df.loc[row_index, "actual_output"] = generated_answer


    except Exception as error:
        error_text = str(error)

        error_reasons[qa_df.loc[row_index, "test_id"]] = {
            "contextual_precision_reason": error_text,
            "answer_relevancy_reason": error_text
        }

        with open(error_reason_file, "w", encoding="utf-8") as file:
            json.dump(error_reasons, file, indent=4)


        print("Error:", qa_df.loc[row_index, "test_id"], error_text)



    # Small delay to reduce free-tier rate-limit issues.
    time.sleep(60)



# Show Contextual Precision result.
display(
    qa_df[
        [
            "test_id",
            "question",
            "gold_answer",
            "actual_output"
        ]
    ]
)

In [ ]:
## 10. Contextual Recall Check using DeepEval Trace

# This section runs only Contextual Recall.
#
# Contextual Recall checks whether the retrieved top-k documents
# contain enough information from the expected/gold answer.
#
# We keep this separate because Contextual Recall creates larger JSON output
# and can fail more easily when combined with other metrics.

from deepeval.evaluate import ErrorConfig, AsyncConfig
from deepeval.dataset import Golden, EvaluationDataset
from deepeval.metrics import ContextualRecallMetric

import json
import time



# Create DeepEval Contextual Recall metric.
# Groq is used as the evaluation judge through our custom wrapper.
contextual_recall_metric = ContextualRecallMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True,
    verbose_mode=True
)


# Create Golden test cases from qa_df.
# Each Golden contains:
# - input: original question
# - expected_output: gold_answer / expected answer
goldens = []

for row_index, row in qa_df.iterrows():
    goldens.append(
        Golden(
            input=row["question"],
            expected_output=row["gold_answer"]
        )
    )


# Create DeepEval evaluation dataset.
evaluation_dataset = EvaluationDataset(
    goldens=goldens
)


# Store only Contextual Recall error reasons when errors happen.
contextual_recall_error_reason_file = "/content/contextual_recall_error_reasons.json"
contextual_recall_error_reasons = {}


for row_index, golden in enumerate(
    evaluation_dataset.evals_iterator(
        metrics=[
            contextual_recall_metric
        ],
        error_config=ErrorConfig(ignore_errors=True),
        async_config=AsyncConfig(run_async=False)
    )
):

    try:
        # Run traced RAG pipeline for this test case.
        # This creates retrieval_context inside DeepEval trace.
        run_traced_rag_pipeline(
            question=golden.input,
            expected_output=golden.expected_output,
            top_k=2
        )

    except Exception as error:
        error_text = str(error)

        contextual_recall_error_reasons[qa_df.loc[row_index, "test_id"]] = {
            "contextual_recall_reason": error_text
        }

        with open(contextual_recall_error_reason_file, "w", encoding="utf-8") as file:
            json.dump(contextual_recall_error_reasons, file, indent=4)


        print("Contextual Recall Error:", qa_df.loc[row_index, "test_id"], error_text)

    # Small delay to reduce free-tier rate-limit issues.
    time.sleep(60)


# Show Contextual Recall result.
display(
    qa_df[
        [
            "test_id",
            "question",
            "gold_answer",
            "actual_output"
        ]
    ]
)

In [ ]:
# Save and download final RAG metric evaluation result

# This CSV includes:
# - generated demo RAG answers
# - DeepEval traced metric results
# - direct measure debug results for missing traced scores

#contextual_precision_output_file = "rag_contextual_precision_tracing_result.csv"
rag_metrics_output_file = "rag_contextual_precision_recall_tracing_result.csv"


#qa_df.to_csv(contextual_precision_output_file, index=False)

contextual_precision_result_df = qa_df[
    [
        "test_id",
        "question",
        "expected_doc_ids",
        "gold_answer",
        "actual_output"
    ]
]

#contextual_precision_result_df.to_csv(contextual_precision_output_file, index=False)

#print("Saved:", contextual_precision_output_file)


from google.colab import files

#files.download(contextual_precision_output_file)


contextual_precision_result_df.to_csv(rag_metrics_output_file, index=False)

print("Saved:", rag_metrics_output_file)

files.download(rag_metrics_output_file)